# Custom BPE Tokenizer Analysis

This notebook trains the from-scratch tokenizer, inspects merge rules and vocabulary behavior, and visualizes token frequency distribution.

In [ ]:
from pathlib import Path
import sys
from collections import Counter

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from tokenizer import BPETokenizer

## Load the Sample Corpus

In [ ]:
corpus_path = PROJECT_ROOT / 'data' / 'sample_corpus.txt'
corpus = corpus_path.read_text(encoding='utf-8').splitlines()
len(corpus), corpus[:3]

## Train the Tokenizer

In [ ]:
tokenizer = BPETokenizer(vocab_size=180, min_frequency=2, lowercase=True)
tokenizer.fit(corpus)

len(tokenizer.get_vocab()), len(tokenizer.merges)

## Inspect Vocabulary Growth

In [ ]:
vocab = tokenizer.get_vocab()
list(vocab.items())[:25]

## Show Top Merge Rules

In [ ]:
for i, pair in enumerate(tokenizer.merges[:20], start=1):
    merged = ''.join(pair)
    print(f'{i:02d}. {pair[0]!r} + {pair[1]!r} -> {merged!r}')

## Encode and Decode Sample Text

In [ ]:
sample = 'Tokenization is important for NLP systems.'
ids = tokenizer.encode(sample)
decoded = tokenizer.decode(ids)
ids, decoded

## Compare Character Length and BPE Token Length

In [ ]:
for text in corpus[:5]:
    char_count = len(text.replace(' ', ''))
    bpe_count = len(tokenizer.encode(text, add_special_tokens=False))
    print(f'chars={char_count:3d} bpe_tokens={bpe_count:3d} | {text[:70]}')

## Visualize Token Frequency Distribution

In [ ]:
import matplotlib.pyplot as plt

token_counts = Counter()
for line in corpus:
    token_counts.update(tokenizer.id_to_token(i) for i in tokenizer.encode(line, add_special_tokens=False))

top_tokens = token_counts.most_common(20)
labels = [token for token, _ in top_tokens]
values = [count for _, count in top_tokens]

plt.figure(figsize=(12, 5))
plt.bar(labels, values)
plt.xticks(rotation=60, ha='right')
plt.title('Top Token Frequencies')
plt.ylabel('Count')
plt.tight_layout()